In [26]:
# ==========================================
# SLA BREACH PREDICTION (FINAL COMPLETE CODE)
# ==========================================

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# ==========================================
# FILE PATHS
# ==========================================

train_path = r"C:\Users\ASUS\OneDrive\Desktop\hackerthorn\train.csv"
test_path  = r"C:\Users\ASUS\OneDrive\Desktop\hackerthorn\test.csv"

# ==========================================
# LOAD DATA
# ==========================================

df_train = pd.read_csv(train_path)
df_test  = pd.read_csv(test_path)

print("✅ Data Loaded")

# Convert breach to 0/1
df_train['breach'] = df_train['breach'].apply(lambda x: 1 if x in [True,1,'True','Yes'] else 0)
df_test['breach']  = df_test['breach'].apply(lambda x: 1 if x in [True,1,'True','Yes'] else 0)

# ==========================================
# FEATURE ENGINEERING
# ==========================================

def engineer(df):
    df = df.copy()
    
    df['priority_score'] = df['priority'].map({'Low':1,'Medium':2,'High':3}).fillna(2)
    
    df['time_used'] = df['estimated_time'] - df['time_remaining']
    df['completion_percentage'] = (df['time_used']/(df['estimated_time']+1))*100
    df['time_ratio'] = df['time_remaining']/(df['estimated_time']+1)
    
    df['urgency_score'] = df['priority_score']*(1/(df['time_ratio']+0.01))
    df['risk_score'] = df['priority_score']*(1-df['time_ratio'])
    
    df['critical_deadline'] = (df['time_remaining'] < 20).astype(int)
    df['high_risk'] = ((df['priority_score']>=3) & (df['time_ratio']<0.3)).astype(int)
    
    return df

df_train = engineer(df_train)
df_test  = engineer(df_test)

# ==========================================
# FEATURES
# ==========================================

features = [
    'priority_score','estimated_time','time_remaining',
    'completion_percentage','time_ratio','urgency_score',
    'risk_score','critical_deadline','high_risk','assigned_to'
]

# ==========================================
# PREPARE DATA
# ==========================================

X_train = df_train[features].copy()
y_train = df_train['breach']

X_test  = df_test[features].copy()
y_test  = df_test['breach']

# Encode categorical
le = LabelEncoder()
X_train['assigned_to'] = le.fit_transform(X_train['assigned_to'].astype(str))

def safe_encode(val):
    return le.transform([val])[0] if val in le.classes_ else 0

X_test['assigned_to'] = X_test['assigned_to'].apply(safe_encode)

# Clean + scale
imp = SimpleImputer(strategy='median')
scaler = StandardScaler()

X_train = scaler.fit_transform(imp.fit_transform(X_train))
X_test  = scaler.transform(imp.transform(X_test))

# ==========================================
# TRAIN MODEL
# ==========================================

model = RandomForestClassifier(n_estimators=300, max_depth=15, class_weight='balanced')
model.fit(X_train, y_train)

print("✅ Model trained")

# ==========================================
# PREDICTION
# ==========================================

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

results = df_test.copy()
results['predicted_breach'] = y_pred
results['probability'] = y_proba

# ==========================================
# SLA SAFE / UNSAFE
# ==========================================

results['SLA_Status'] = results['predicted_breach'].map({
    0: '✅ SLA SAFE',
    1: '⚠️ SLA UNSAFE'
})

# ==========================================
# PERFORMANCE
# ==========================================

print("\n📊 PERFORMANCE:")
print("Accuracy:", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall:", recall_score(y_test,y_pred))
print("F1:", f1_score(y_test,y_pred))
print("ROC-AUC:", roc_auc_score(y_test,y_proba))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test,y_pred))

# ==========================================
# SLA SUMMARY
# ==========================================

print("\n===== SLA STATUS SUMMARY =====")

safe_count = (results['predicted_breach']==0).sum()
unsafe_count = (results['predicted_breach']==1).sum()

total = len(results)

print(f"✅ SAFE:   {safe_count} ({safe_count/total*100:.2f}%)")
print(f"⚠️ UNSAFE: {unsafe_count} ({unsafe_count/total*100:.2f}%)")

# ==========================================
# SAMPLE OUTPUT
# ==========================================

print("\n===== SAMPLE RESULTS =====")

cols = ['priority','estimated_time','time_remaining','assigned_to','SLA_Status','probability']
print(results[cols].head(20).to_string(index=False))

# ==========================================
# ADVANCED ANALYSIS
# ==========================================

print("\n🔍 ADVANCED ANALYSIS")

# Error
results['error'] = "Correct"
results.loc[(results['breach']==0)&(results['predicted_breach']==1),'error']="False Positive"
results.loc[(results['breach']==1)&(results['predicted_breach']==0),'error']="False Negative"

print("\nError Breakdown:")
print(results['error'].value_counts())

# Cost
actual_loss = results[results['breach']==1]['penalty_cost'].sum()
predicted_loss = results[results['predicted_breach']==1]['penalty_cost'].sum()
preventable = results[(results['breach']==1)&(results['predicted_breach']==1)]['penalty_cost'].sum()

print("\n💰 COST ANALYSIS:")
print("Actual Loss:", actual_loss)
print("Predicted Risk:", predicted_loss)
print("Preventable:", preventable)

# High risk
high_risk = results[results['probability']>0.8]
print("\n⚠️ High Risk Tasks:", len(high_risk))

# ==========================================
# SAVE OUTPUT
# ==========================================

results.to_csv("sla_results_with_status.csv",index=False)
pickle.dump(model,open("sla_model.pkl","wb"))

print("\n✅ Saved: sla_results_with_status.csv")
print("✅ Saved: sla_model.pkl")

print("\n🚀 DONE!")

✅ Data Loaded
✅ Model trained

📊 PERFORMANCE:
Accuracy: 0.783
Precision: 0.25268817204301075
Recall: 0.376
F1: 0.3022508038585209
ROC-AUC: 0.5337417142857142

Confusion Matrix:
[[736 139]
 [ 78  47]]

===== SLA STATUS SUMMARY =====
✅ SAFE:   814 (81.40%)
⚠️ UNSAFE: 186 (18.60%)

===== SAMPLE RESULTS =====
priority  estimated_time  time_remaining assigned_to    SLA_Status  probability
     Low             188              40           G    ✅ SLA SAFE     0.478326
    High              69              90           G    ✅ SLA SAFE     0.110899
    High              81              54           G    ✅ SLA SAFE     0.422192
    High             134             118           G    ✅ SLA SAFE     0.383390
  Medium             115              39           C    ✅ SLA SAFE     0.490314
  Medium             126             116           D    ✅ SLA SAFE     0.393499
    High              93             117           B    ✅ SLA SAFE     0.335645
     Low             181              77           G 

In [32]:
# ==========================================
# 🤖 GEN AI INTEGRATION (SLA INTELLIGENCE LAYER)
# ==========================================

# Install if not installed:
# pip install openai

from openai import OpenAI
import time

# 🔑 Add your API key here
client = OpenAI(api_key="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")

# ==========================================
# GEN AI FUNCTION
# ==========================================

def generate_ai_insight(row):
    """
    Generate explanation + action + business impact
    """

    prompt = f"""
    You are an enterprise SLA optimization AI.

    Task Details:
    - Priority: {row.get('priority', 'Unknown')}
    - Estimated Time: {row.get('estimated_time', 'Unknown')}
    - Time Remaining: {row.get('time_remaining', 'Unknown')}
    - Breach Probability: {row.get('probability', 0):.2f}
    - Penalty Cost: {row.get('penalty_cost', 0)}

    Provide:
    1. Reason for SLA risk
    2. Recommended corrective action
    3. Business impact in simple terms
    Keep it short and professional.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=150
        )

        return response.choices[0].message.content.strip()

    except Exception as e:
        return f"Error generating insight: {str(e)}"


# ==========================================
# APPLY GEN AI ONLY TO RISKY TASKS
# ==========================================

print("\n🤖 Generating AI insights for risky tasks...")

results['AI_Insight'] = ""

count = 0

for i, row in results.iterrows():
    
    # Only for unsafe tasks
    if row['predicted_breach'] == 1:
        
        insight = generate_ai_insight(row)
        results.at[i, 'AI_Insight'] = insight
        
        count += 1
        
        # Avoid API rate limit
        time.sleep(0.5)

        if count % 10 == 0:
            print(f"Processed {count} risky tasks...")

print(f"\n✅ AI insights generated for {count} risky tasks")

# ==========================================
# SAVE FINAL OUTPUT
# ==========================================

output_file = r"C:\Users\ASUS\OneDrive\Desktop\hackerthorn\sla_results_with_status.csv"
results.to_csv(output_file, index=False)

print(f"\n📁 Saved file: {output_file}")

# ==========================================
# SHOW SAMPLE OUTPUT
# ==========================================

print("\n===== SAMPLE AI OUTPUT =====")

sample_cols = [
    'priority',
    'estimated_time',
    'time_remaining',
    'SLA_Status',
    'probability',
    'AI_Insight'
]

print(results[sample_cols].head(10).to_string(index=False))


🤖 Generating AI insights for risky tasks...
Processed 10 risky tasks...
Processed 20 risky tasks...
Processed 30 risky tasks...
Processed 40 risky tasks...
Processed 50 risky tasks...
Processed 60 risky tasks...
Processed 70 risky tasks...
Processed 80 risky tasks...
Processed 90 risky tasks...
Processed 100 risky tasks...
Processed 110 risky tasks...
Processed 120 risky tasks...
Processed 130 risky tasks...
Processed 140 risky tasks...
Processed 150 risky tasks...
Processed 160 risky tasks...
Processed 170 risky tasks...
Processed 180 risky tasks...
Processed 190 risky tasks...
Processed 200 risky tasks...
Processed 210 risky tasks...
Processed 220 risky tasks...
Processed 230 risky tasks...
Processed 240 risky tasks...
Processed 250 risky tasks...
Processed 260 risky tasks...
Processed 270 risky tasks...
Processed 280 risky tasks...
Processed 290 risky tasks...
Processed 300 risky tasks...
Processed 310 risky tasks...
Processed 320 risky tasks...
Processed 330 risky tasks...
Process